In [57]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yanishelali/dataset1/WA_Fn-UseC_-Telco-Customer-Churn.csv


# Jour 3 — Phase 0 : Setup des algorithmes

Objectif :
- repartir du dataset nettoyé
- préparer les données pour les algorithmes de Machine Learning
- vérifier que le pipeline fonctionne

Nous utilisons le dataset Telco Customer Churn nettoyé au Jour 2.

In [58]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/yanishelali/dataset1/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape :", df.shape)
df.head()

Shape : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [59]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [60]:
df = df.drop(columns=["customerID"])

In [61]:
binary_map = {"Yes": 1, "No": 0}

binary_cols = df.columns[df.nunique() == 2]

for col in binary_cols:
    df[col] = df[col].map(binary_map)

df = pd.get_dummies(df, drop_first=True)

In [62]:
df = df.drop(columns=["TotalCharges"])

In [63]:
print("Shape :", df.shape)
print(df.dtypes)

Shape : (7043, 30)
gender                                   float64
SeniorCitizen                            float64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
Churn                                      int64
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No internet service             bool
T

## Conclusion Phase 0

Le dataset est maintenant :

- nettoyé
- encodé
- sans variables redondantes
- prêt à être utilisé avec des algorithmes

On peut maintenant tester différents modèles de Machine Learning.

# Jour 3 — Phase A : L'Arène des algorithmes

Objectif :
Comparer plusieurs modèles de Machine Learning sur le dataset Telco Customer Churn.

Modèles testés :
- Régression logistique
- Arbre de décision
- Random Forest
- KNN

Critère d’évaluation :
- Accuracy

But :
Identifier le modèle le plus performant.


In [64]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [65]:
import pandas as pd

# Chargement
df = pd.read_csv("/kaggle/input/datasets/yanishelali/dataset1/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Fix TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Supprimer ID
df = df.drop(columns=["customerID"])

In [66]:
# Colonnes catégorielles
categorical_cols = df.select_dtypes(include=["object"]).columns

# Mapping binary Yes/No UNIQUEMENT
for col in categorical_cols:
    if set(df[col].unique()) == {"Yes", "No"}:
        df[col] = df[col].map({"Yes": 1, "No": 0})

# One-hot pour le reste
df = pd.get_dummies(df, drop_first=True)

In [67]:
# Variable redondante
df = df.drop(columns=["TotalCharges"])

In [68]:
print("NaN restants :", df.isna().sum().sum())
print("Shape :", df.shape)

NaN restants : 0
Shape : (7043, 30)


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
## Fonction d'entraînement et d'évaluation

In [ ]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

In [ ]:
## Comparaison des modèles

In [ ]:
modeles = {
    "Régression Logistique": LogisticRegression(max_iter=300),
    "Arbre de décision": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}

resultats = []

for nom, modele in modeles.items():
    score = entrainer_et_evaluer(
        modele,
        X_train,
        X_test,
        y_train,
        y_test
    )
    resultats.append((nom, score))

In [ ]:
resultats = sorted(resultats, key=lambda x: x[1], reverse=True)

print("=== Classement des modèles ===\n")

for i, (nom, score) in enumerate(resultats, start=1):
    print(f"{i}. {nom} : {score:.2%}")

In [ ]:
import matplotlib.pyplot as plt

noms = [nom for nom, score in resultats]
scores = [score for nom, score in resultats]

plt.figure()
plt.bar(noms, scores)

plt.title("Comparaison des modèles")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)

plt.show()

In [ ]:
## Analyse des résultats

- Random Forest est généralement le modèle le plus performant
- Logistic Regression offre un bon compromis entre performance et simplicité
- KNN dépend fortement des données
- l'Arbre de décision est moins stable

Conclusion :

Random Forest est le modèle le plus adapté pour ce dataset.

Cependant, Logistic Regression reste intéressante pour l'interprétation.